# Family A Parser Validation

**Objective:** Validate parser detection and Family A parsing logic against real BVC workbook samples.

Expected Family A sheets: Cours, Bid, Ask, Volume MC, Quantité MC  
Expected Family B sheets: Data, Indicateurs

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add project root to path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.parsers.parser_factory import detect_sheet_family, SheetKind
from src.parsers.simple_sheet_parser import parse_simple_sheet

print(f'Project root: {ROOT}')

Project root: /home/yass/Desktop/DSS_CMR


## Step 1: Load workbook and inspect sheet names

In [2]:
# Use sample workbook for testing
wb_path = ROOT / 'Données Marché Boursier_Projet_IA_.xlsx'
xl = pd.ExcelFile(wb_path, engine='openpyxl')

print(f'Workbook: {wb_path.name}')
print(f'Total sheets: {len(xl.sheet_names)}\n')
for i, sheet in enumerate(xl.sheet_names, 1):
    print(f'{i:2d}. {sheet}')

Workbook: Données Marché Boursier_Projet_IA_.xlsx
Total sheets: 7

 1. Data
 2. Cours
 3. Bid
 4. Ask
 5. Quantité MC
 6. Volume MC
 7. Indicateurs


## Step 2: Test detection logic on each sheet

In [3]:
detection_results = []

for sheet_name in xl.sheet_names:
    # Load first 20 rows to detect family
    df = pd.read_excel(wb_path, sheet_name=sheet_name, nrows=20, header=None)
    detected = detect_sheet_family(df)
    
    detection_results.append({
        'Sheet': sheet_name,
        'Detected': detected.value,
        'Rows': len(df),
        'Cols': len(df.columns)
    })

detection_df = pd.DataFrame(detection_results)
print('\nDetection Results:')
print(detection_df.to_string(index=False))


Detection Results:
      Sheet Detected  Rows  Cols
       Data family_b    20  1412
      Cours family_a    20    84
        Bid family_a    20    84
        Ask family_a    20    84
Quantité MC family_a    20    84
  Volume MC family_a    20    84
Indicateurs  unknown     3    44


## Step 3: Inspect structure of each sheet type

In [4]:
# Inspect first Family A sheet (Cours)
cours_df = pd.read_excel(wb_path, sheet_name='Cours', nrows=15, header=None)
print('=== Cours (Expected Family A) ===')
print(f'Shape: {cours_df.shape}')
print('\nFirst 15 rows, first 6 columns:')
print(cours_df.iloc[:, :6])

=== Cours (Expected Family A) ===
Shape: (15, 84)

First 15 rows, first 6 columns:
                      0                    1                    2  \
0              Code AMC                 1229                 1211   
1             CODE ISIN         MA0000012296         MA0000012114   
2               LIBELLE                 AFMA  AFRIC INDUSTRIES SA   
3                   NaN  MA0000012296,XX,CAS  MA0000012114,XX,CAS   
4                   NaN               AFMA P         Afric Indus.   
5                   NaN                  VAL                  VAL   
6   2018-12-31 00:00:00                  990                  270   
7   2019-01-02 00:00:00                  990                  270   
8   2019-01-03 00:00:00                  990                  286   
9   2019-01-04 00:00:00                  980                  286   
10  2019-01-07 00:00:00                  980                  286   
11  2019-01-08 00:00:00                  980                  286   
12  2019-01-09 00:00

In [5]:
# Inspect Data sheet (Expected Family B)
data_df = pd.read_excel(wb_path, sheet_name='Data', nrows=30, header=None)
print('=== Data (Expected Family B) ===')
print(f'Shape: {data_df.shape}')
print('\nFirst 30 rows, first 6 columns:')
print(data_df.iloc[:, :6])

=== Data (Expected Family B) ===
Shape: (30, 1412)

First 30 rows, first 6 columns:
                      0                    1                    2  \
0              Code AMC                 1229                 1211   
1             CODE ISIN         MA0000012296         MA0000012114   
2               LIBELLE                 AFMA  AFRIC INDUSTRIES SA   
3                   NaN  MA0000012296,XX,CAS  MA0000012114,XX,CAS   
4                   NaN               AFMA P                  NaN   
5                   NaN           ALTHIGHMID            ALTLOWMID   
6                   NaN                  NaN                  NaN   
7   2026-07-21 00:00:00                  NaN                  NaN   
8   2026-07-20 00:00:00                  NaN                  NaN   
9   2026-07-17 00:00:00                  NaN                  NaN   
10  2026-07-16 00:00:00                  NaN                  NaN   
11  2026-07-15 00:00:00                  NaN                  NaN   
12  2026-07-14 00:0

## Step 4: Test Family A parser on market sheets

In [6]:
# Parse Cours sheet
print('=== Parsing Cours ===' )
try:
    cours_parsed = parse_simple_sheet(str(wb_path), 'Cours')
    print(f'✓ Success: {len(cours_parsed)} records')
    print(f'  Columns: {list(cours_parsed.columns)}')
    print(f'  Unique CODE_ISIN: {cours_parsed["CODE_ISIN"].nunique()}')
    print(f'  Unique Dates: {cours_parsed["Date"].nunique()}')
    print(f'  Date range: {cours_parsed["Date"].min()} to {cours_parsed["Date"].max()}')
    print('\nSample records:')
    print(cours_parsed.head(10))
except Exception as e:
    print(f'✗ Error: {type(e).__name__}: {e}')

=== Parsing Cours ===
✓ Success: 155791 records
  Columns: ['Date', 'CODE_ISIN', 'Company', 'Variable', 'Value']
  Unique CODE_ISIN: 83
  Unique Dates: 1877
  Date range: 2018-12-31 00:00:00 to 2026-07-21 00:00:00

Sample records:
        Date     CODE_ISIN              Company Variable   Value
0 2018-12-31  MA0000012296                 AFMA    Cours   990.0
1 2018-12-31  MA0000012114  AFRIC INDUSTRIES SA    Cours   270.0
2 2018-12-31  MA0000010951         AFRIQUIA GAZ    Cours  3000.0
3 2018-12-31  MA0000010944                 AGMA    Cours  3079.0
4 2018-12-31  MA0000012585              AKDITAL    Cours     NaN
5 2018-12-31  MA0000011819            ALLIANCES    Cours    85.0
6 2018-12-31  MA0000010936   ALUMINIUM DU MAROC    Cours  1565.0
7 2018-12-31  MA0000012460       ARADEI CAPITAL    Cours     NaN
8 2018-12-31  MA0000011710         ATLANTASANAD    Cours    57.0
9 2018-12-31  MA0000012445    ATTIJARIWAFA BANK    Cours   453.0


In [7]:
# Parse all Family A sheets
family_a_sheets = ['Cours', 'Bid', 'Ask', 'Volume MC', 'Quantité MC']
parse_results = {}

for sheet in family_a_sheets:
    print(f'\n=== Parsing {sheet} ===')
    try:
        parsed = parse_simple_sheet(str(wb_path), sheet)
        parse_results[sheet] = parsed
        print(f'✓ Records: {len(parsed)}')
        print(f'  Companies: {parsed["CODE_ISIN"].nunique()}')
        print(f'  Sessions: {parsed["Date"].nunique()}')
        print(f'  Null values: {parsed["Value"].isna().sum()} ({parsed["Value"].isna().mean()*100:.1f}%)')
        print(f'  Variable name: {parsed["Variable"].unique()}')
    except Exception as e:
        print(f'✗ Error: {type(e).__name__}: {e}')
        parse_results[sheet] = None


=== Parsing Cours ===
✓ Records: 155791
  Companies: 83
  Sessions: 1877
  Null values: 24558 (15.8%)
  Variable name: <ArrowStringArray>
['Cours']
Length: 1, dtype: str

=== Parsing Bid ===
✓ Records: 47025
  Companies: 75
  Sessions: 627
  Null values: 9417 (20.0%)
  Variable name: <ArrowStringArray>
['Bid']
Length: 1, dtype: str

=== Parsing Ask ===
✓ Records: 47025
  Companies: 75
  Sessions: 627
  Null values: 2533 (5.4%)
  Variable name: <ArrowStringArray>
['Ask']
Length: 1, dtype: str

=== Parsing Volume MC ===
✓ Records: 155708
  Companies: 83
  Sessions: 1876
  Null values: 59990 (38.5%)
  Variable name: <ArrowStringArray>
['Volume MC']
Length: 1, dtype: str

=== Parsing Quantité MC ===
✓ Records: 155708
  Companies: 83
  Sessions: 1876
  Null values: 59988 (38.5%)
  Variable name: <ArrowStringArray>
['Quantité MC']
Length: 1, dtype: str


## Step 5: Validate CODE ISIN extraction

In [8]:
# Check CODE ISIN patterns
if 'Cours' in parse_results and parse_results['Cours'] is not None:
    cours_df = parse_results['Cours']
    unique_isins = cours_df['CODE_ISIN'].unique()
    
    print(f'Total unique CODE ISIN values: {len(unique_isins)}')
    print('\nSample CODE ISIN values:')
    for isin in sorted(unique_isins)[:10]:
        print(f'  {isin}')
    
    # Check for invalid ISINs
    invalid = cours_df[~cours_df['CODE_ISIN'].str.startswith('MA', na=False)]['CODE_ISIN'].unique()
    if len(invalid) > 0:
        print(f'\n⚠ Warning: {len(invalid)} invalid CODE ISIN values:')
        for isin in invalid[:5]:
            print(f'  {isin}')

Total unique CODE ISIN values: 83

Sample CODE ISIN values:
  MA0000010019
  MA0000010035
  MA0000010068
  MA0000010340
  MA0000010357
  MA0000010365
  MA0000010381
  MA0000010415
  MA0000010506
  MA0000010571


## Step 6: Check for data quality issues

In [9]:
# Data quality checks
print('=== Data Quality Summary ===')
for sheet, df in parse_results.items():
    if df is not None:
        print(f'\n{sheet}:')
        print(f'  Missing dates: {df["Date"].isna().sum()}')
        print(f'  Missing CODE_ISIN: {df["CODE_ISIN"].isna().sum()}')
        print(f'  Empty CODE_ISIN: {(df["CODE_ISIN"] == "").sum()}')
        print(f'  Missing values: {df["Value"].isna().sum()}')
        
        # Check for duplicate keys
        dups = df.groupby(['Date', 'CODE_ISIN']).size()
        dup_count = (dups > 1).sum()
        if dup_count > 0:
            print(f'  ⚠ Duplicate Date×CODE_ISIN: {dup_count}')

=== Data Quality Summary ===

Cours:
  Missing dates: 0
  Missing CODE_ISIN: 0
  Empty CODE_ISIN: 0
  Missing values: 24558

Bid:
  Missing dates: 0
  Missing CODE_ISIN: 0
  Empty CODE_ISIN: 0
  Missing values: 9417

Ask:
  Missing dates: 0
  Missing CODE_ISIN: 0
  Empty CODE_ISIN: 0
  Missing values: 2533

Volume MC:
  Missing dates: 0
  Missing CODE_ISIN: 0
  Empty CODE_ISIN: 0
  Missing values: 59990

Quantité MC:
  Missing dates: 0
  Missing CODE_ISIN: 0
  Empty CODE_ISIN: 0
  Missing values: 59988


## Step 7: Verify all sheets have same companies

In [10]:
# Check if all Family A sheets have same set of companies
isin_sets = {}
for sheet, df in parse_results.items():
    if df is not None:
        isin_sets[sheet] = set(df['CODE_ISIN'].unique())

if len(isin_sets) > 1:
    # Compare all sets
    reference_sheet = list(isin_sets.keys())[0]
    reference_set = isin_sets[reference_sheet]
    
    print(f'Reference sheet: {reference_sheet} ({len(reference_set)} companies)\n')
    
    all_match = True
    for sheet, isin_set in isin_sets.items():
        if sheet == reference_sheet:
            continue
        
        if isin_set == reference_set:
            print(f'✓ {sheet}: matches reference ({len(isin_set)} companies)')
        else:
            all_match = False
            only_in_ref = reference_set - isin_set
            only_in_current = isin_set - reference_set
            print(f'✗ {sheet}: MISMATCH')
            print(f'  Only in {reference_sheet}: {len(only_in_ref)}')
            print(f'  Only in {sheet}: {len(only_in_current)}')
    
    if all_match:
        print('\n✓ All Family A sheets have consistent company sets')
    else:
        print('\n⚠ Company sets are inconsistent across sheets')

Reference sheet: Cours (83 companies)

✗ Bid: MISMATCH
  Only in Cours: 8
  Only in Bid: 0
✗ Ask: MISMATCH
  Only in Cours: 8
  Only in Ask: 0
✗ Volume MC: MISMATCH
  Only in Cours: 1
  Only in Volume MC: 1
✗ Quantité MC: MISMATCH
  Only in Cours: 1
  Only in Quantité MC: 1

⚠ Company sets are inconsistent across sheets


## Step 8: Summary and issues

In [11]:
print('=== PARSER VALIDATION SUMMARY ===')
print(f'\nWorkbook: {wb_path.name}')
print(f'Total sheets: {len(xl.sheet_names)}')
print(f'\nFamily A sheets expected: {family_a_sheets}')
print(f'Successfully parsed: {len([v for v in parse_results.values() if v is not None])}')
print(f'Failed: {len([v for v in parse_results.values() if v is None])}')

print('\n--- Detection accuracy ---')
for _, row in detection_df.iterrows():
    sheet = row['Sheet']
    detected = row['Detected']
    
    if sheet in family_a_sheets:
        expected = 'family_a'
        status = '✓' if detected == expected else '✗'
    elif sheet in ['Data', 'Indicateurs']:
        expected = 'family_b'
        status = '✓' if detected == expected else '✗'
    else:
        expected = 'unknown'
        status = '?'
    
    print(f'{status} {sheet:20s} -> {detected:20s} (expected: {expected})')

print('\n--- Next steps ---')
print('1. Fix any detection mismatches')
print('2. Fix any parsing errors')
print('3. Handle data quality issues')
print('4. Move to notebook 04 (normalization) when validated')

=== PARSER VALIDATION SUMMARY ===

Workbook: Données Marché Boursier_Projet_IA_.xlsx
Total sheets: 7

Family A sheets expected: ['Cours', 'Bid', 'Ask', 'Volume MC', 'Quantité MC']
Successfully parsed: 5
Failed: 0

--- Detection accuracy ---
✓ Data                 -> family_b             (expected: family_b)
✓ Cours                -> family_a             (expected: family_a)
✓ Bid                  -> family_a             (expected: family_a)
✓ Ask                  -> family_a             (expected: family_a)
✓ Quantité MC          -> family_a             (expected: family_a)
✓ Volume MC            -> family_a             (expected: family_a)
✗ Indicateurs          -> unknown              (expected: family_b)

--- Next steps ---
1. Fix any detection mismatches
2. Fix any parsing errors
3. Handle data quality issues
4. Move to notebook 04 (normalization) when validated
